# NB02_fPINN — Training on the fractional plant: integer MLP / HYBRID / PINN vs fractional PINN and HYBRID

**Purpose.** Second step of the fractional-PINN paper. NB01_fPINN showed that least squares cannot identify α once the τ_u residual of notebook 18 is present (s = 1). Here the networks of notebook 18 are retrained, with **exactly its protocol** (`train_ff_model`: 13-64-64 SiLU, Adam lr 1e-3, weight decay 1e-5, batch 2048, ReduceLROnPlateau, early stopping on validation MSE, patience 18, 150 epochs, gradient clipping 1.0), on the plant

`τ = γ(m_L)·Y(q, q̈, p)·θ0 + s·τ_u(q, q̇) + B·D^α q`,   `s = 1`, `α ∈ {0.3, 0.5, 0.7, 0.9}`, `B = 0.05 N m s^α` per joint,

and compared with two new models that add a fractional branch with **learnable α (shared) and B (per joint)**:

- **fPINN:** `τ̂ = Y(q, q̈, p)·θ̂(x) + B̂·D^α̂ q`
- **fHYBRID:** `τ̂ = γ·Y·θ0 + B̂·D^α̂ q + σ_r·net(x)`

**How D^α̂ is made differentiable.** `D^α q` is precomputed for every sample (L1 scheme on the logged 2.5 ms signal, full memory since t = 0) on a grid of α (step 0.025) and linearly interpolated in α inside the network, so the gradient reaches α at the cost of one gather per batch. The label uses the generator rule (0.5 ms, full memory), as in NB01_fPINN.

**One switch.** `QUICK = True` is a fast check of the whole pipeline (~2 min: 6 networks at 3 epochs, the nb-18 check, one sweep case). `QUICK = False` runs **the whole experiment set of the paper** in one go, with nothing else to edit:

| Block | α | Networks | Trajectories | Seeds | Count |
|---|---|---|---|---|---|
| Main study | 0.5 | MLP, HYBRID, PINN, fPINN, fHYBRID | 35, 175 | 0, 1, 2 | 30 |
| α sweep | 0.3, 0.7, 0.9 | HYBRID, PINN, fPINN, fHYBRID | 175 | 0 | 12 |
| Check vs notebook 18 | — | PINN (original plant) | 35 | 0 | 1 |

**Expected runtime (Ryzen 7 PRO 5850U, 16 GB):** measured here at ~2 s/epoch with 35 trajectories and ~11 s/epoch with 175, the upper bound is ≈ 14 h (printed at start); early stopping and 8 cores should bring it to roughly 8–12 h, so run it overnight. Every finished network is cached in `CACHE_DIR` and its weights saved, so an interruption only loses the network in progress: rerun and it resumes. RAM ≈ 2 GB, printed after every network (needs `psutil`, which Anaconda includes); memory is released between networks. Needs NumPy, SciPy, pandas, matplotlib and PyTorch (CPU), as notebook 18.

**Outputs:** `NB02_fPINN_TrainingLog.csv` (one row per network: α, kind, size, seed, test and Adept torque RMSE, α̂, B̂, epochs, time), `NB02_fPINN_Summary.csv`, `fig_nb02_fpinn_alpha.png`. Keep the `NB02_fPINN_<kind>_<size>traj_seed<k>_a<α>_e150.npz` weight files (one per network): NB03_fPINN loads them.

**Built-in check.** `VALIDATE_NB18 = True` first retrains PINN, 35 trajectories, seed 0, on the **original** plant of notebook 18 (no fractional term) and compares its test RMSE with the saved network of notebook 18 (0.01539 N m); with the same PyTorch version as notebook 18 it should match closely.

In [ ]:
import os, gc, time, copy, random, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK       = True                 # True: fast check (~2 min). False: the whole experiment set of the paper (see header).
SEVERITY    = 1.0
B_FRAC      = 0.05                 # [N m s^α] per joint
KINDS       = ("MLP", "HYBRID", "PINN", "fPINN", "fHYBRID")
ALPHA_MAIN  = 0.5                  # main study: every network, 35 and 175 trajectories, seeds 0-2
MAIN_SIZES, MAIN_SEEDS = (35, 175), (0, 1, 2)
ALPHA_SWEEP = (0.3, 0.7, 0.9)      # α sweep: networks with physics, 175 trajectories, seed 0
SWEEP_KINDS, SWEEP_SIZES, SWEEP_SEEDS = ("PINN", "HYBRID", "fPINN", "fHYBRID"), (175,), (0,)
TRAIN_CFG   = dict(batch_size=2048, lr=1e-3, weight_decay=1e-5, max_epochs=150,
                   patience=18, lr_patience=6, log_every=25)                       # notebook 18
ALPHA_INIT  = 0.8                  # initial guess for α (the true value is not used)
VALIDATE_NB18 = True               # retrain PINN 35 traj seed 0 on the original plant and compare with nb 18
WEIGHTS_DIR = "."                  # folder with ff18_PINN_35traj_seed0_e150.npz (only for the check message)
CACHE_DIR   = "NB02_fPINN_cache"   # finished runs are saved here and reloaded on rerun (like REUSE_SAVED in nb 18)
if QUICK:
    MAIN_SIZES, MAIN_SEEDS = (35,), (0,)
    ALPHA_SWEEP, SWEEP_KINDS, SWEEP_SIZES, SWEEP_SEEDS = (0.3,), ("fPINN",), (35,), (0,)
    TRAIN_CFG = dict(TRAIN_CFG, max_epochs=3, log_every=1)
EXPERIMENTS = ([(ALPHA_MAIN, k, n, sd) for n in MAIN_SIZES for sd in MAIN_SEEDS for k in KINDS]
               + [(a, k, n, sd) for a in ALPHA_SWEEP for n in SWEEP_SIZES for sd in SWEEP_SEEDS for k in SWEEP_KINDS])
ALPHAS_ALL = (ALPHA_MAIN,) + tuple(ALPHA_SWEEP)
# -----------------------------------------------------------------------------------------------
FS = 2000; dt = 1.0/FS; SUBSAMPLE = 5
DATA_SEED, N_VAL_TRAJ, N_TEST_TRAJ, N_TRAIN_MAX = 12345, 20, 20, 175
ALPHA_GRID = np.round(np.arange(0.05, 1.0001, 0.025), 3)
import pickle; os.makedirs(CACHE_DIR, exist_ok=True)
torch.set_num_threads(min(8, max(1, (os.cpu_count() or 2)//2)))      # physical cores; avoids per-thread buffers
print("QUICK" if QUICK else "FULL", f"| {len(EXPERIMENTS)} networks + {int(VALIDATE_NB18)} check | epochs {TRAIN_CFG['max_epochs']} | torch {torch.__version__}")
_sec = {35: 2.2, 175: 11.0}                                   # s/epoch measured on a 4-core cloud CPU
_est = sum(_sec[n] for _, _, n, _ in EXPERIMENTS)*TRAIN_CFG["max_epochs"] + VALIDATE_NB18*_sec[35]*TRAIN_CFG["max_epochs"]
print(f"rough upper bound for the training time: {_est/3600:.1f} h (early stopping can shorten it)")

## Robot model and unmodeled dynamics (notebook 18)

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)
def gamma_payload(m): return (M_BASE + np.asarray(m, float)) / M_BASE

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def regressor_batch(q, a, P):
    n = len(q); s, c = np.sin(q), np.cos(q); x, y, z = P[:, 0], P[:, 1], P[:, 2]
    Y = np.zeros((n, 3, 14)); Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y

def tau_unmodeled_batch(q, qd, sev):
    u = UNMODELED
    return sev*(u["b_v"]*qd + u["f_c"]*np.tanh(qd/u["eps"])
                + u["c_cor"]*(qd**2*np.cos(q) + np.roll(qd, -1, axis=1)*np.roll(qd, -2, axis=1))
                + u["k_pos"]*np.sin(u["w_pos"]*q))

def caputo_l1(x, h, alpha, window=None):
    x = np.asarray(x, float); N = len(x); K = N if window is None else min(N, int(window))
    k = np.arange(K); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, b[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

## Data: notebook 18 trajectories (same RNG), fractional labels, D^α features on the α grid

In [ ]:
def random_trajectory_full(rng, n_wp=6, v_range=(0.03, 0.8), t_min=0.3):     # RNG use identical to nb 18
    W = np.column_stack([rng.uniform(-0.12, 0.12, n_wp), rng.uniform(-0.12, 0.12, n_wp), rng.uniform(0.49, 0.56, n_wp)])
    dist = np.linalg.norm(np.diff(W, axis=0), axis=1)
    v = np.exp(rng.uniform(np.log(v_range[0]), np.log(v_range[1]), n_wp - 1))
    dur = np.maximum(dist / v, t_min); m_load = float(rng.integers(0, 7))
    edges = np.concatenate([[0.0], np.cumsum(dur)]); t_g = np.arange(0.0, edges[-1], dt)
    seg = np.clip(np.searchsorted(edges, t_g, side="right") - 1, 0, n_wp - 2)
    ts = (t_g - edges[seg]) / dur[seg]; s = 6*ts**5 - 15*ts**4 + 10*ts**3
    P = W[seg] + (W[seg + 1] - W[seg])*s[:, None]
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m_load)

def log_and_label(T, margin_s=0.05):
    """Same samples as nb 18 (slice(mg, N-mg, SUBSAMPLE)). Returns nominal torque, τ_u (s = 1), the true
    fractional torque (generator: 0.5 ms, full memory) and the D^α grid features (identifier: logged 2.5 ms q)."""
    q, qd, qdd, P, m = T["q"], T["qd"], T["qdd"], T["P"], T["m"]
    nom = gamma_payload(m)*np.einsum("nij,j->ni", regressor_batch(q, qdd, P), THETA0)
    frac = {a: B_FRAC*caputo_l1(q, dt, a) for a in ALPHAS_ALL}; tu = tau_unmodeled_batch(q, qd, 1.0)
    mg = int(margin_s/dt); idx = np.arange(0, len(q), SUBSAMPLE); keep = (idx >= mg) & (idx < len(q) - mg)
    h = dt*SUBSAMPLE; ql = q[idx]
    G = np.stack([caputo_l1(ql, h, a)[keep] for a in ALPHA_GRID], axis=1).astype(np.float32)   # (n, n_alpha, 3)
    i = idx[keep]
    return dict(X=np.column_stack([q[i], qd[i], qdd[i], np.full(len(i), m), P[i]]), nom=nom[i], tu=tu[i], G=G, **{f"frac_{a}": f[i] for a, f in frac.items()})

def build(n_traj, rng):
    parts = [log_and_label(random_trajectory_full(rng)) for _ in range(n_traj)]
    D = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    D["traj"] = np.concatenate([np.full(len(p["X"]), j) for j, p in enumerate(parts)]); return D

t0 = time.time(); rng_data = np.random.default_rng(DATA_SEED)
DATA_TRAIN_ALL = build(N_TRAIN_MAX, rng_data); DATA_VAL = build(N_VAL_TRAJ, rng_data); DATA_TEST = build(N_TEST_TRAJ, rng_data)
print(f"data in {time.time()-t0:.0f} s | train {len(DATA_TRAIN_ALL['X']):,} val {len(DATA_VAL['X']):,} test {len(DATA_TEST['X']):,}")

def labels(D, plant, alpha=None):
    """plant 'nb18': original plant of notebook 18; 'frac': + B D^α q with the given α."""
    y = D["nom"] + SEVERITY*D["tu"] + (D[f"frac_{alpha}"] if plant == "frac" else 0.0)
    return dict(X=D["X"], y=y, base=D["nom"], G=D["G"], traj=D["traj"])
def subset(D, size):
    if size >= int(D["traj"].max()) + 1: return D                     # full set: no copy
    keep = D["traj"] < size; return {k: v[keep] for k, v in D.items()}

def ram_gb():
    """Resident memory of this kernel in GB (needs psutil; NaN otherwise)."""
    try:
        import psutil; return psutil.Process().memory_info().rss/1e9
    except Exception: return float("nan")
for a in ALPHAS_ALL: print("alpha %.1f: rms fractional torque %.4f N m" % (a, np.sqrt(np.mean(DATA_TRAIN_ALL[f"frac_{a}"]**2))))
print("rms τ_u %.4f N m" % np.sqrt(np.mean(DATA_TRAIN_ALL["tu"]**2)))

## Training (train_ff_model of notebook 18 + fractional branch)

Only additions to notebook 18: the fractional branch (`fPINN`, `fHYBRID`), its two parameters kept out of weight decay, and the export of α̂, B̂. The PINN regressor is written as in notebook 18.

In [ ]:
GA = torch.tensor(ALPHA_GRID, dtype=torch.float32); DA = float(ALPHA_GRID[1] - ALPHA_GRID[0])
def interp_D(G, alpha):
    """G: (b, n_alpha, 3); differentiable linear interpolation in alpha."""
    u = torch.clamp((alpha - GA[0]) / DA, 0.0, len(ALPHA_GRID) - 1.000001)
    i0 = torch.floor(u).long(); w = u - i0
    return (1 - w)*G[:, i0, :] + w*G[:, i0 + 1, :]

def train_ff_model(kind, d_tr, d_va, seed, cfg=TRAIN_CFG):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    x_mean, x_std = d_tr["X"].mean(0), d_tr["X"].std(0) + 1e-8
    y_mean, y_std = d_tr["y"].mean(0), d_tr["y"].std(0)
    r_std = (d_tr["y"] - d_tr["base"]).std(0) + 1e-8
    PHYS_IDX = [0, 1, 2, 6, 7, 8, 10, 11, 12]
    frac = kind.startswith("f"); core = kind[1:] if frac else kind

    class FFNet(nn.Module):
        def __init__(self):
            super().__init__()
            n_out = 14 if core == "PINN" else 3
            self.net = nn.Sequential(nn.Linear(13, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, n_out))
            for name_, arr in [("y_mean", y_mean), ("y_std", y_std), ("r_std", r_std)]:
                self.register_buffer(name_, torch.tensor(arr, dtype=torch.float32))
            if core == "PINN":
                with torch.no_grad():
                    self.net[-1].weight.mul_(0.1); self.net[-1].bias.copy_(torch.tensor(THETA0, dtype=torch.float32))
            if frac:
                a0 = (ALPHA_INIT - 0.05)/0.925
                self.a_raw = nn.Parameter(torch.tensor(np.log(a0/(1 - a0)), dtype=torch.float32))
                self.B = nn.Parameter(torch.zeros(3))
        def alpha(self): return 0.05 + 0.925*torch.sigmoid(self.a_raw)
        def forward(self, xs, xp, base, G):
            out = self.net(xs)
            if core == "MLP": tau = out*self.y_std + self.y_mean
            elif core == "HYBRID": tau = base + out*self.r_std
            else:
                q, qdd = xp[:, 0:3], xp[:, 3:6]; x, y, z = xp[:, 6], xp[:, 7], xp[:, 8]
                th = [out[:, i] for i in range(14)]
                t1 = (th[0]*qdd[:, 0] - th[1]*x*torch.sin(q[:, 0]) - th[2]*y*torch.sin(q[:, 0])
                      + th[3]*z*torch.cos(q[:, 0]) + th[4]*torch.cos(q[:, 0]) + th[5]*torch.sin(q[:, 0]))
                t2 = (th[0]*qdd[:, 1] - th[6]*x*torch.sin(q[:, 1]) - th[7]*y*torch.sin(q[:, 1])
                      + th[8]*z*torch.cos(q[:, 1]) + th[4]*torch.cos(q[:, 1]) + th[9]*torch.sin(q[:, 1]))
                t3 = (th[0]*qdd[:, 2] - th[10]*x*torch.sin(q[:, 2]) - th[11]*y*torch.sin(q[:, 2])
                      + th[12]*z*torch.cos(q[:, 2]) + th[4]*torch.cos(q[:, 2]) + th[13]*torch.sin(q[:, 2]))
                tau = torch.stack([t1, t2, t3], dim=1)
            if frac: tau = tau + self.B*interp_D(G, self.alpha())
            return tau

    def to_tensors(d):
        xs = torch.tensor((d["X"] - x_mean)/x_std, dtype=torch.float32)
        xp = torch.tensor(d["X"][:, PHYS_IDX], dtype=torch.float32)
        G = torch.from_numpy(d["G"]) if frac else torch.zeros(len(xs), 1, 1)
        return TensorDataset(xs, xp, torch.tensor(d["base"], dtype=torch.float32), G, torch.tensor(d["y"], dtype=torch.float32))

    train_loader = DataLoader(to_tensors(d_tr), batch_size=cfg["batch_size"], shuffle=True, generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(to_tensors(d_va), batch_size=cfg["batch_size"], shuffle=False)
    model = FFNet()
    groups = [{"params": model.net.parameters(), "weight_decay": cfg["weight_decay"]}]
    if frac: groups.append({"params": [model.a_raw, model.B], "weight_decay": 0.0})
    opt = torch.optim.Adam(groups, lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=cfg["lr_patience"])
    loss_fn = nn.MSELoss(); best, best_state, wait = np.inf, None, 0; t_tr = time.time(); hist = []
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        for xs, xp, base, G, y in train_loader:
            opt.zero_grad(); loss = loss_fn(model(xs, xp, base, G), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for xs, xp, base, G, y in val_loader:
                vs += loss_fn(model(xs, xp, base, G), y).item()*len(y); vn += len(y)
        val = vs/vn; sched.step(val)
        if frac: hist.append((epoch, model.alpha().item(), *model.B.detach().numpy().tolist()))
        if val < best - 1e-12: best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
        else: wait += 1
        if epoch % cfg["log_every"] == 0 or epoch == 1 or wait >= cfg["patience"]:
            extra = f" | α {model.alpha().item():.3f} B {np.round(model.B.detach().numpy(), 4)}" if frac else ""
            print(f"    [{kind} {d_tr['traj'].max()+1}traj seed{seed}] epoch {epoch:3d} | val MSE {val:.3e} | best {best:.3e}{extra} | {time.time()-t_tr:.0f} s", flush=True)
        if wait >= cfg["patience"]: break
    del train_loader, val_loader, opt, sched                           # free the training tensors before returning
    model.load_state_dict(best_state); model.eval()
    info = dict(epochs=epoch, best_val_mse=best, train_s=time.time() - t_tr)
    lins = [m_ for m_ in model.net if isinstance(m_, nn.Linear)]
    info["export"] = dict(kind=kind, **{k: t_.detach().numpy().astype(float) for k, t_ in
                          zip(("W1", "b1", "W2", "b2", "W3", "b3"), [t for l_ in lins for t in (l_.weight, l_.bias)])},
                          x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std, r_std=r_std,
                          alpha=model.alpha().item() if frac else 0.0,
                          B=model.B.detach().numpy().astype(float) if frac else np.zeros(3))
    if frac: info.update(alpha_hat=model.alpha().item(), B_hat=model.B.detach().numpy().astype(float))
    return model, to_tensors, info, hist

def rmse_t(model, to_tensors, d):
    xs, xp, base, G, y = to_tensors(d).tensors
    with torch.no_grad(): return float(torch.sqrt(torch.mean((model(xs, xp, base, G) - y)**2)))

## Check: the training code reproduces notebook 18 (original plant, PINN, 35 trajectories, seed 0)

In [ ]:
if VALIDATE_NB18:
    f_chk = os.path.join(CACHE_DIR, f"check_nb18_PINN_35_seed0_e{TRAIN_CFG['max_epochs']}.pkl")
    if os.path.exists(f_chk): chk = pickle.load(open(f_chk, "rb"))
    else:
        tr, va, te = (labels(subset(DATA_TRAIN_ALL, 35), "nb18"), labels(DATA_VAL, "nb18"), labels(DATA_TEST, "nb18"))
        m_chk, tt_chk, info_chk, _ = train_ff_model("PINN", tr, va, 0)
        chk = dict(test=rmse_t(m_chk, tt_chk, te), epochs=info_chk["epochs"]); pickle.dump(chk, open(f_chk, "wb"))
    print(f"retrained here: test RMSE {chk['test']:.5f} N m ({chk['epochs']} epochs) | notebook 18 saved network: 0.01539 N m" + (" (QUICK: 3 epochs, not comparable)" if QUICK else ""))

## Adept references (NB01) for the on-policy torque check (0 and 6 kg)

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05
SPEED_SCALES = (6.4, 3.2, 1.6)
def adept_full(scale, m):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]; P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m)
REFS = {}
for sc in SPEED_SCALES:
    for m in (0.0, 6.0):
        R = log_and_label(adept_full(sc, m), margin_s=0.0); R["traj"] = np.zeros(len(R["X"]), int); REFS[(sc, m)] = R

## Run on the fractional plant

In [ ]:
rows, hists = [], {}; t_all = time.time(); E = TRAIN_CFG["max_epochs"]
for j, (alpha, kind, size, seed) in enumerate(EXPERIMENTS, 1):
    f_c = os.path.join(CACHE_DIR, f"{kind}_{size}traj_seed{seed}_a{alpha}_e{E}.pkl")
    f_w = f"NB02_fPINN_{kind}_{size}traj_seed{seed}_a{alpha}_e{E}.npz"          # weights for NB03_fPINN
    if os.path.exists(f_c) and os.path.exists(f_w):
        row, hist = pickle.load(open(f_c, "rb")); rows.append(row)
        if len(hist): hists[(alpha, kind, size, seed)] = hist
        print(f"[{j}/{len(EXPERIMENTS)}] α {alpha} {kind:8s} {size:3d} traj seed {seed}: loaded from cache"); continue
    TR, VA, TE = labels(subset(DATA_TRAIN_ALL, size), "frac", alpha), labels(DATA_VAL, "frac", alpha), labels(DATA_TEST, "frac", alpha)
    model, tt, info, hist = train_ff_model(kind, TR, VA, seed)
    row = dict(**{"alpha true": alpha}, Kind=kind, Trajectories=size, Seed=seed, Epochs=info["epochs"],
               **{"Train time (s)": info["train_s"], "Val RMSE (N m)": rmse_t(model, tt, VA), "Test RMSE (N m)": rmse_t(model, tt, TE)})
    for (sc, m), Rf in REFS.items(): row[f"Adept {sc} {m:.0f} kg (N m)"] = rmse_t(model, tt, labels(Rf, "frac", alpha))
    if "alpha_hat" in info:
        row["alpha hat"] = info["alpha_hat"]; row["B hat mean"] = float(info["B_hat"].mean()); row["B hat"] = [round(float(b), 4) for b in info["B_hat"]]
        hists[(alpha, kind, size, seed)] = np.array(hist)
    np.savez(f_w, **info["export"])                          # NumPy export (as nb 18) + α̂, B̂; read by NB03_fPINN
    pickle.dump((row, np.array(hist)), open(f_c, "wb"))
    rows.append(row); pd.DataFrame(rows).to_csv("NB02_fPINN_TrainingLog.csv", index=False)
    print(f"[{j}/{len(EXPERIMENTS)}] α {alpha} {kind:8s} {size:3d} traj seed {seed}: test {row['Test RMSE (N m)']:.5f} N m"
          + (f" | α̂ {row['alpha hat']:.3f} B̂ {row['B hat']}" if "alpha hat" in row else "")
          + f" | {(time.time()-t_all)/60:.1f} min | RAM {ram_gb():.2f} GB", flush=True)
    del model, tt, info, hist, TR, VA, TE; gc.collect()                # nothing of a finished network stays in RAM
df = pd.DataFrame(rows)

## Summary

In [ ]:
pd.set_option("display.width", 250)
df.to_csv("NB02_fPINN_TrainingLog.csv", index=False)
print(f"true B = {B_FRAC} N m s^α, s = {SEVERITY}")
cols = ["Test RMSE (N m)"] + [f"Adept {sc} {m:.0f} kg (N m)" for sc in SPEED_SCALES for m in (0.0, 6.0)]
summ = df.groupby(["alpha true", "Kind", "Trajectories"])[cols].agg(["mean", "std"])
summ.to_csv("NB02_fPINN_Summary.csv")
print("Torque RMSE (mN m), mean over seeds:"); print((1e3*df.groupby(["alpha true", "Kind", "Trajectories"])[cols].mean()).round(2).to_string())
if "alpha hat" in df:
    print("\nIdentified fractional parameters:")
    print(df.dropna(subset=["alpha hat"])[["alpha true", "Kind", "Trajectories", "Seed", "alpha hat", "B hat mean", "B hat"]].to_string(index=False))

## Figures: α̂ during training (main α) and identified α̂ vs true α

In [ ]:
if hists:
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
    for (alpha, kind, size, seed), h in hists.items():
        if alpha == ALPHA_MAIN:
            axes[0].plot(h[:, 0], h[:, 1], lw=0.8, ls="-" if kind == "fPINN" else "--", label=f"{kind} {size} traj s{seed}")
    axes[0].axhline(ALPHA_MAIN, color="k", lw=0.6, ls=":"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("α̂")
    axes[0].set_title(f"Training, true α = {ALPHA_MAIN}", fontsize=8); axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=5, ncol=2)
    fr = df.dropna(subset=["alpha hat"])
    for (kind, size), g in fr.groupby(["Kind", "Trajectories"]):
        st = g.groupby("alpha true")["alpha hat"].agg(["mean", "std"]).reset_index()
        axes[1].errorbar(st["alpha true"], st["mean"], yerr=st["std"].fillna(0), marker="o" if kind == "fPINN" else "s",
                         ms=4, capsize=2, lw=1, label=f"{kind} {size} traj")
    axes[1].plot([0, 1], [0, 1], "k:", lw=0.6); axes[1].set_xlim(0.2, 1.0); axes[1].set_ylim(0.2, 1.0)
    axes[1].set_xlabel("True α"); axes[1].set_ylabel("Identified α̂"); axes[1].set_title("α sweep", fontsize=8)
    axes[1].grid(True, alpha=0.3); axes[1].legend(fontsize=6)
    fig.tight_layout(); fig.savefig("fig_nb02_fpinn_alpha.png", dpi=200); fig.savefig("fig_nb02_fpinn_alpha.pdf"); plt.show()